In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

In [2]:
# Load data
data = pd.read_csv('data/adult.csv', header=None)

# Assign column names
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

# Keep only desired features
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'hours-per-week', 'capital-gain', 'capital-loss', 'income']]

# Drop rows with missing values
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()

# One-Hot encode categorical values
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation'])

In [3]:
# Split testing and training
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

# Apply standard scaling to data
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [4]:
# Create and fit baseline model
model = LogisticRegression(random_state=21).fit(X_train, y_train)

with open("models/baseline.pkl", "wb") as file:
    pickle.dump(model, file)

In [5]:
y_pred = model.predict(X_test) 
y_prob = model.predict_proba(X_test)[:, 1] 

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=3))

auc = roc_auc_score(y_test, y_prob) 
print(f"AUC: {auc:.3f}")

report = classification_report(y_test, y_pred, digits=3, output_dict=True)
baseline_metrics = [accuracy, report[' >50K']['precision'], report[' >50K']['recall'], report[' >50K']['f1-score'], auc]

Accuracy: 0.844
Confusion Matrix:
 [[9628  763]
 [1392 2027]]
              precision    recall  f1-score   support

       <=50K      0.874     0.927     0.899     10391
        >50K      0.727     0.593     0.653      3419

    accuracy                          0.844     13810
   macro avg      0.800     0.760     0.776     13810
weighted avg      0.837     0.844     0.838     13810

AUC: 0.898


In [6]:
# Create and fit MLP model
model = MLPClassifier(random_state=21, max_iter=1000, hidden_layer_sizes=(50,), alpha=0.05).fit(X_train, y_train)

with open("models/mlp.pkl", "wb") as file:
    pickle.dump(model, file)

In [7]:
y_pred = model.predict(X_test) 
y_prob = model.predict_proba(X_test)[:, 1] 

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=3))

auc = roc_auc_score(y_test, y_prob) 
print(f"AUC: {auc:.3f}")

report = classification_report(y_test, y_pred, digits=3, output_dict=True)
mlp_metrics = [accuracy, report[' >50K']['precision'], report[' >50K']['recall'], report[' >50K']['f1-score'], auc]

Accuracy: 0.852
Confusion Matrix:
 [[9618  773]
 [1264 2155]]
              precision    recall  f1-score   support

       <=50K      0.884     0.926     0.904     10391
        >50K      0.736     0.630     0.679      3419

    accuracy                          0.852     13810
   macro avg      0.810     0.778     0.792     13810
weighted avg      0.847     0.852     0.848     13810

AUC: 0.907


In [8]:
data = {
    "Metric": ["Accuracy", "Precision (>50K)", "Recall (>50K)", "F1 (>50K)", "AUC"],
    "Baseline": [baseline_metrics[0], baseline_metrics[1], baseline_metrics[2], baseline_metrics[3], baseline_metrics[4]],
    "MLP": [mlp_metrics[0], mlp_metrics[1], mlp_metrics[2], mlp_metrics[3], mlp_metrics[4]],
    "MLP - Baseline": [(mlp_metrics[0] - baseline_metrics[0]), (mlp_metrics[1] - baseline_metrics[1]), (mlp_metrics[2] - baseline_metrics[2]), (mlp_metrics[3] - baseline_metrics[3]), (mlp_metrics[4] - baseline_metrics[4])]
}

pd.DataFrame(data).style.format(
    subset=["Baseline", "MLP", "MLP - Baseline"], formatter="{:.3f}"
).hide(axis="index")

Metric,Baseline,MLP,MLP - Baseline
Accuracy,0.844,0.852,0.009
Precision (>50K),0.727,0.736,0.009
Recall (>50K),0.593,0.630,0.037
F1 (>50K),0.653,0.679,0.026
AUC,0.898,0.907,0.009
